# Old Permic OCR — Archival Manuscript Synthesis & Final-Model Trial

**Layer 4 / Notebook 3:** this notebook creates 20 reproducible, full-page manuscript documents using the project's own glyph renderer. It accepts user-supplied parchment/document images, applies interleaved Old Permic glyphs, faded/raised/engraved materials, controlled occlusion, perspective, noise, signature and seal marks, and writes YOLO labels plus an audit JSON for every page.

> These are **synthetic research artifacts**, clearly marked in metadata as generated samples. They must not be presented as authentic historical documents. The purpose is to stress-test OCR on difficult visual conditions and to compare another script's document-like appearance with Old Permic glyph recognition.


In [ ]:
# Cell 01 — install/import (Colab or local Jupyter)
from pathlib import Path
import sys, os, glob, json, re
REPO_DIR = Path('/content/ocroldpermic') if Path('/content/ocroldpermic').exists() else Path.cwd()
if str(REPO_DIR / 'lib') not in sys.path:
    sys.path.insert(0, str(REPO_DIR / 'lib'))
print('Repository:', REPO_DIR)


In [ ]:
# Cell 02 — synchronize the active code and latest model checkpoints
import shutil, subprocess

REPO_URL = "https://github.com/Emran025/ocroldpermic"
CODE_BRANCH = "archival-manuscript-synthesis"
CHECKPOINT_BRANCH = "colab-checkpoints"  # candidate training artifacts
RESULT_BRANCH = "colab-results"  # published production artifacts
CHECKPOINT_REPO_DIR = Path('/content/final-checkpoints')
RESULT_REPO_DIR = Path('/content/final-results')

def sync_branch(url, branch, target):
    target = Path(target)
    if (target / '.git').is_dir():
        subprocess.run(['git', '-C', str(target), 'fetch', 'origin', branch], check=True)
        subprocess.run(['git', '-C', str(target), 'checkout', '-B', branch, f'origin/{branch}'], check=True)
    else:
        if target.exists():
            shutil.rmtree(target)
        target.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', '--branch', branch, '--depth', '1', url, str(target)], check=True)

sync_branch(REPO_URL, CODE_BRANCH, REPO_DIR)
sync_branch(REPO_URL, CHECKPOINT_BRANCH, CHECKPOINT_REPO_DIR)
# The final trial must consume only a promoted production release.
sync_branch(REPO_URL, RESULT_BRANCH, RESULT_REPO_DIR)
if str(REPO_DIR / 'lib') not in sys.path:
    sys.path.insert(0, str(REPO_DIR / 'lib'))
print(f'Code checkout: {REPO_DIR} @ {CODE_BRANCH}')
print(f'Checkpoint checkout: {CHECKPOINT_REPO_DIR} @ {CHECKPOINT_BRANCH} (candidate)')
print(f'Published results checkout: {RESULT_REPO_DIR} @ {RESULT_BRANCH}')


## Inputs and reproducibility

Place one or more background photographs in `BACKGROUND_DIR`. The generator cycles through them deterministically. If the directory is empty, it creates a Codex Runicus-inspired procedural parchment from the measured reference profile: warm ochre palette, edge wear, stains, fibres, and laid-line cadence. Change `SEED` to create a new controlled set, while keeping the same seed reproduces the exact set.


In [ ]:
# Cell 03 — configuration
from historical_glyph_studio import GlyphStudio
from historical_glyph_studio.document_generator import DocumentSpec, generate_documents

GLYPH_ROOT = REPO_DIR / 'font' / 'svg'
BACKGROUND_DIR = REPO_DIR / 'user_backgrounds'  # upload your parchment/document images here
OUTPUT_DIR = Path('/content/archival_documents_20')
SEED = 20260907
DOCUMENT_COUNT = 20
BACKGROUND_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

backgrounds = sorted([p for p in BACKGROUND_DIR.rglob('*') if p.suffix.lower() in {'.png','.jpg','.jpeg','.webp','.tif','.tiff'}])
studio = GlyphStudio(glyph_root=GLYPH_ROOT)
print(studio.repository_summary())
print(f'Backgrounds found: {len(backgrounds)}')


## Visual generation

The page layout intentionally resembles a difficult manuscript: multiple baselines, tight character spacing, per-glyph rotation and perspective, faded dark pigment, mild occlusion, uneven ink, and separate signature/seal marks. The annotations contain only glyph boxes, so the seal and signature do not become false OCR classes.


In [ ]:
# Cell 04 — generate the 20-document showcase/evaluation set
specs = [
    DocumentSpec(
        document_id=i + 1,
        seed=SEED + i,
        lines=14 + (i % 3),
        min_chars=18,
        max_chars=30,
        material=('faded_black' if i % 3 else 'engraved'),
        handwriting_family='01_Original_Handwriting',
        handwriting_style='Original',
        background='codex_runicus_procedural',
        include_signature=True,
        include_seal=(i % 2 == 0),
    )
    for i in range(DOCUMENT_COUNT)
]
paths = generate_documents(studio, specs, OUTPUT_DIR, backgrounds)
print(f'Generated {len(paths)} pages in {OUTPUT_DIR}')
print('Example:', paths[0] if paths else 'none')


In [ ]:
# Cell 05 — inspect a contact sheet and verify artifact counts
from PIL import Image, ImageDraw
from IPython.display import display
imgs = [Image.open(p).resize((210, 285)) for p in paths[:20]]
sheet = Image.new('RGB', (210 * 5, 305 * 4), (40, 30, 20))
draw = ImageDraw.Draw(sheet)
for i, img in enumerate(imgs):
    x, y = (i % 5) * 210, (i // 5) * 305
    sheet.paste(img, (x, y))
    draw.text((x + 5, y + 287), f'document_{i+1:02d}', fill=(255, 235, 190))
sheet_path = OUTPUT_DIR / 'contact_sheet.png'
sheet.save(sheet_path)
display(sheet)
print('Images:', len(list(OUTPUT_DIR.glob('document_*.png'))))
print('Labels:', len(list(OUTPUT_DIR.glob('document_*.txt'))))
print('Metadata:', len(list(OUTPUT_DIR.glob('document_*.json'))))


## Select the latest available trained model

This cell does not assume Stage 12. It searches common Colab/checkpoint/release locations, reads numeric stage IDs from paths and metadata, and selects the highest available stage. If no trained artifact exists yet, it stops with an actionable message instead of silently using an arbitrary model.


In [ ]:
# Cell 06 — discover and validate the published model from colab-results
import json, re, hashlib

def _stage_from_path(path):
    matches = re.findall(r'(?:stage[_-]?(\d+)|s(\d+))', str(path).lower())
    return max([int(a or b) for a,b in matches], default=-1)

def discover_latest_model(results_root):
    """Resolve the model through artifacts/published/latest.json only."""
    root = Path(results_root)
    pointer_path = root / 'artifacts' / 'published' / 'latest.json'
    if not pointer_path.is_file():
        raise FileNotFoundError(
            'No artifacts/published/latest.json exists on colab-results. '
            'Run the training/publishing notebook first.'
        )
    pointer = json.loads(pointer_path.read_text(encoding='utf-8'))
    release_path = pointer.get('release_path')
    if not release_path:
        raise RuntimeError('latest.json has no release_path')
    release_file = root / release_path
    if not release_file.is_file():
        release_file = root / 'artifacts' / 'published' / release_path
    if not release_file.is_file():
        raise FileNotFoundError(f'Published release metadata is missing: {release_file}')
    release = json.loads(release_file.read_text(encoding='utf-8'))
    if release.get('publication_status') != 'published':
        raise RuntimeError(f'Latest artifact is not published: {release.get("publication_status")}')
    release_dir = release_file.parent
    weight_path = release.get('web_weight', {}).get('path')
    if not weight_path:
        assets = release.get('assets', [])
        weight_path = next((a.get('path') for a in assets if a.get('kind') == 'pytorch_weight'), None)
    if not weight_path:
        raise RuntimeError('Published release has no PyTorch weight path')
    artifact = release_dir / weight_path
    if not artifact.is_file():
        raise FileNotFoundError(f'Published model weight is missing: {artifact}')
    stage_match = re.search(r'(?:^|[-_])s(\d+)(?:[-_]|$)', release.get('release_id', '').lower())
    stage = int(stage_match.group(1)) if stage_match else -1
    if stage < 0:
        stage_match = re.search(r'stage[-_]?(\d+)', release.get('release_id', '').lower())
        stage = int(stage_match.group(1)) if stage_match else -1
    if stage < 0:
        raise RuntimeError(f'Published release has no parseable stage: {release.get("release_id")}')
    return stage, release, artifact

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''): h.update(block)
    return h.hexdigest()

latest_stage, latest_release, latest_model = discover_latest_model(RESULT_REPO_DIR)
if latest_stage < 0:
    raise RuntimeError(f'Latest published artifact {latest_model} has no stage metadata; refusing an untracked model.')
model_sha256 = sha256_file(latest_model)
print(f'Latest published stage: {latest_stage:02d}')
print(f'Selected published model: {latest_model}')
print(f'Model SHA-256: {model_sha256}')


## Final-model experiment

The evaluation cell is deliberately adapter-based. It supports Ultralytics checkpoints (`.pt`) and exported ONNX/`.ocrpkg` artifacts through the project's runtime packaging conventions. It writes predictions separately from human-readable transcriptions and never overwrites the raw labels or source pages.


In [ ]:
# Cell 07 — final end-to-end trial on the 20 generated pages
PREDICTION_DIR = OUTPUT_DIR / 'predictions'
PREDICTION_DIR.mkdir(exist_ok=True)

def validate_final_inputs(image_paths):
    if len(image_paths) != DOCUMENT_COUNT:
        raise RuntimeError(f'Expected exactly {DOCUMENT_COUNT} final pages, found {len(image_paths)}')
    missing=[]
    for image in image_paths:
        label=Path(image).with_suffix('.txt')
        metadata=Path(image).with_suffix('.json')
        if not Path(image).is_file() or not label.is_file() or not metadata.is_file():
            missing.append(str(image))
    if missing:
        raise RuntimeError(f'Final page artifacts are incomplete: {missing[:3]}')
    for image in image_paths:
        for line in Path(image).with_suffix('.txt').read_text(encoding='utf-8').splitlines():
            fields=line.split()
            if len(fields) != 5:
                raise RuntimeError(f'Invalid final-page label in {image}: {line!r}')
            cls, cx, cy, width, height = fields
            if not (0 <= int(cls) < len(studio.available_class_names())):
                raise RuntimeError(f'Final-page class id {cls} is outside the model alphabet')
            values=[float(cx), float(cy), float(width), float(height)]
            if not (0 <= values[0] <= 1 and 0 <= values[1] <= 1 and 0 < values[2] <= 1 and 0 < values[3] <= 1):
                raise RuntimeError(f'Invalid normalized bounding box in {image}: {line!r}')

def validate_model_alphabet(model_path, expected_classes):
    suffix=model_path.suffix.lower()
    if suffix == '.pt':
        try:
            from ultralytics import YOLO
            model=YOLO(str(model_path))
            names=getattr(model, 'names', {}) or {}
            if len(names) != expected_classes:
                raise RuntimeError(f'Model exposes {len(names)} classes, expected {expected_classes}')
            return {'model_classes': len(names), 'contract': 'ultralytics names match'}
        except ImportError:
            print('Ultralytics unavailable: model class-count inspection deferred to runtime.')
            return {'model_classes': None, 'contract': 'deferred; ultralytics unavailable'}
    if suffix in {'.ocrpkg', '.onnx'}:
        return {'model_classes': None, 'contract': 'runtime adapter required'}
    raise ValueError(f'Unsupported model artifact: {model_path}')

def run_latest_model(model_path, image_paths, output_dir):
    suffix=model_path.suffix.lower()
    if suffix == '.pt':
        try:
            from ultralytics import YOLO
        except ImportError as exc:
            raise ImportError('Install ultralytics to evaluate a .pt checkpoint: pip install ultralytics') from exc
        model=YOLO(str(model_path))
        results=model.predict(source=[str(p) for p in image_paths], project=str(output_dir), name='latest_stage', exist_ok=True, save=True, save_txt=True, conf=0.20, verbose=False)
        prediction_files=list((output_dir/'latest_stage'/'labels').glob('*.txt')) if (output_dir/'latest_stage'/'labels').exists() else []
        return {'backend':'ultralytics', 'model':str(model_path), 'images':len(results), 'prediction_files':len(prediction_files)}
    if suffix in {'.onnx','.ocrpkg'}:
        return {'backend':'onnx_runtime_adapter', 'model':str(model_path), 'images':len(image_paths), 'status':'runtime adapter required'}
    raise ValueError(f'Unsupported model artifact: {model_path}')

validate_final_inputs(paths)
model_contract=validate_model_alphabet(latest_model, len(studio.available_class_names()))
experiment=run_latest_model(latest_model, paths, PREDICTION_DIR)
final_status = experiment.get('images') == DOCUMENT_COUNT
(Path(OUTPUT_DIR)/'final_experiment.json').write_text(json.dumps({
    'status':'PASS' if final_status else 'FAIL',
    'code_branch':CODE_BRANCH, 'checkpoint_branch':CHECKPOINT_BRANCH, 'results_branch':RESULT_BRANCH,
    'latest_stage':latest_stage, 'model':str(latest_model), 'model_sha256':model_sha256,
    'model_contract':model_contract, 'generated_documents':len(paths),
    'expected_documents':DOCUMENT_COUNT, 'experiment':experiment,
}, ensure_ascii=False, indent=2), encoding='utf-8')
if not final_status:
    raise RuntimeError(f'Final trial failed: {experiment}')
print(json.dumps(experiment, ensure_ascii=False, indent=2))


In [ ]:
# Cell 08 — final audit summary
manifest=json.loads((OUTPUT_DIR/'final_experiment.json').read_text())
print('Final integration-test manifest:')
print(json.dumps(manifest, ensure_ascii=False, indent=2))
assert manifest['status']=='PASS'
assert manifest['generated_documents']==manifest['expected_documents']==20
print('PASS: the third notebook generated 20 pages and exercised the newest checkpoint-branch model.')
print('Artifacts are in:', OUTPUT_DIR)
